# Accelerating Tensor Operations in Computational Solid Mechanics

**Optimised contraction strategies and vectorised execution from deep learning, applied to mechanics**

This notebook benchmarks three canonical tensor operations in computational solid mechanics using techniques borrowed from the deep learning ecosystem:

| Benchmark | Operation | Key Technique | Speedup |
|-----------|-----------|---------------|---------|
| **Anisotropic Rotation** | $C'_{pqrs} = R_{pi} R_{qj} C_{ijkl} R_{rk} R_{sl}$ | Contraction path optimisation + `vmap` | ~377× |
| **J2 Plasticity Tangent** | Algorithmic tangent $\mathbb{C}_{ep}$ | Branchless vectorisation + JIT | ~181× |
| **High-Order Matrix-Free** | $\mathbf{f}_e = \sum_g \mathbf{B}_g^T \mathbf{D} \mathbf{B}_g \mathbf{u}_e$ | Fused einsum + sum-factorisation | ~8–29× |

**Environment:** Google Colab, CPU-only, float64 arithmetic.  
**Dependencies:** NumPy, JAX, opt\_einsum, matplotlib (auto-installed below).

> **Paper:** *Accelerating tensor operations in computational solid mechanics through optimised contraction strategies and vectorised execution*  
> P G Kubendran Amos, National Institute of Technology Tiruchirappalli

## How to Run

1. Open this notebook in [Google Colab](https://colab.research.google.com/)
2. Go to **Runtime → Run all** (or press `Ctrl+F9`)
3. The entire benchmark takes approximately **1–2 minutes** on a Colab CPU
4. Figures are saved as PDF/PNG in the working directory

> **Note:** JAX is configured for CPU with `float64` enabled. No GPU is required.

## Setup: Dependencies & Utilities

The cell below installs required packages (if not present), configures JAX for CPU with `float64`, and defines timing/reporting utilities.

**Key libraries:**
- **NumPy** — baseline array operations
- **opt\_einsum** — optimal tensor contraction path finding ([Smith & Gray, JOSS 2018](https://doi.org/10.21105/joss.00753))
- **JAX** — `vmap` for automatic vectorisation, `jit` for XLA compilation ([Bradbury et al., 2018](https://github.com/google/jax))
- **matplotlib** — publication-quality figures

In [ ]:
"""
================================================================================
BENCHMARK: Optimized Tensor Contraction Strategies from Deep Learning
         Applied to Computational Solid Mechanics
================================================================================

Cases:
  1. Isotropic low-order linear elasticity  (B^T D B stiffness assembly)
  2. Anisotropic material rotation          (R.R.C.R.R 4th-order rotation)
  3. Nonlinear constitutive tangent update  (J2 plasticity tangent modulus)
  4. High-order matrix-free operator        (p=4 hex, sum-factorization)

Environment: Google Colab, CPU-only
Outputs:  console tables + publication-ready PDF/PNG figures
================================================================================
"""

import subprocess, sys, time

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("=" * 72)
print("  STEP 0 : Installing dependencies")
print("=" * 72)

for pkg in ["opt_einsum", "jax", "jaxlib", "matplotlib"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"  Installing {pkg} ...")
        install(pkg)

import warnings
warnings.filterwarnings("ignore")

import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", True)

import numpy as np
import opt_einsum as oe
import jax.numpy as jnp
from jax import vmap, jit
from timeit import timeit
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for Colab/headless
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

print(f"  NumPy     : {np.__version__}")
print(f"  JAX       : {jax.__version__}")
print(f"  opt_einsum: {oe.__version__}")
print(f"  Platform  : {jax.devices()[0]}")
print(f"  Float64   : {jax.config.jax_enable_x64}")
print()


# ── Utilities ────────────────────────────────────────────────────────────────

def banner(n, title):
    print("\n" + "=" * 72)
    print(f"  CASE {n} : {title}")
    print("=" * 72)

def progress(s):
    print(f"    Running: {s} ... ", end="", flush=True)

def done(t):
    print(f"done  [{t:.4f} s]")

def report(rows, flop_info=None):
    print()
    print(f"    {'Method':<45} {'Time (s)':>10} {'Speedup':>10} {'Max Error':>12}")
    print(f"    {'-'*45} {'-'*10} {'-'*10} {'-'*12}")
    t0 = rows[0][1]
    for name, t, err in rows:
        sp = t0 / t if t > 0 else float('inf')
        es = f"{err:.2e}" if err is not None else "reference"
        print(f"    {name:<45} {t:>10.4f} {sp:>10.1f}x {es:>12}")
    if flop_info:
        print()
        for l in flop_info:
            print(f"    {l}")
    print()

def time_fn(fn, n_runs=5, n_repeat=3):
    times = []
    for _ in range(n_repeat):
        t = timeit(fn, number=n_runs) / n_runs
        times.append(t)
    return min(times)

def extract_flop_info(path_info):
    lines = []
    for ln in str(path_info[1]).split('\n'):
        s = ln.strip()
        if any(k in s.lower() for k in ['scaling', 'flop', 'speedup']):
            lines.append(s)
    return lines



## Benchmark 1: Anisotropic Material Rotation

**Physical context:** In polycrystalline simulations (e.g., crystal plasticity FEM), the fourth-order elasticity tensor $C_{ijkl}$ must be rotated from the crystal frame to the global frame at every integration point:

$$C'_{pqrs} = R_{pi}\, R_{qj}\, C_{ijkl}\, R_{rk}\, R_{sl}$$

This five-operand contraction over eight indices is evaluated at **10,000 integration points**, each with a unique crystal orientation.

**What is being compared:**
- **(A) Conventional:** `np.einsum('pi,qj,ijkl,rk,sl->pqrs', ...)` in a Python loop — contracts all indices at once, $\mathcal{O}(N^8)$
- **(B) opt\_einsum:** Finds the optimal pairwise contraction order ($\mathcal{O}(N^8) \to \mathcal{O}(N^5)$, from 32,800 to 1,944 FLOPs), but still inside a Python loop
- **(C) JAX vmap+jit:** The optimal sequential path hard-coded + `vmap` over all 10,000 points + XLA compilation

**Key insight:** Path optimisation reduces FLOPs by 17×, but the Python loop overhead negates this when using opt\_einsum in a loop. Only when combined with `vmap` (eliminating the loop) and `jit` (fusing operations) does the full speedup materialise.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CASE 2 : ANISOTROPIC MATERIAL ROTATION
# ══════════════════════════════════════════════════════════════════════════════

def run_case_2():
    banner(2, "Anisotropic Material Rotation (4th-Order Tensor)")
    print("    C'_pqrs = R_pi R_qj C_ijkl R_rk R_sl")
    print("    10000 points, each with unique crystal orientation")
    print()

    NP = 10000
    np.random.seed(123)

    C0 = np.random.randn(3, 3, 3, 3) * 1e9
    C0 = 0.5 * (C0 + C0.transpose(1, 0, 2, 3))
    C0 = 0.5 * (C0 + C0.transpose(0, 1, 3, 2))
    C0 = 0.5 * (C0 + C0.transpose(2, 3, 0, 1))

    R_all = np.zeros((NP, 3, 3))
    for i in range(NP):
        Q, _ = np.linalg.qr(np.random.randn(3, 3))
        if np.linalg.det(Q) < 0: Q[:, 0] *= -1
        R_all[i] = Q

    # (A) Naive
    progress("(A) Conventional np.einsum (per point, no path opt)")
    def naive():
        out = np.zeros((NP, 3, 3, 3, 3))
        for n in range(NP):
            R = R_all[n]
            out[n] = np.einsum('pi,qj,ijkl,rk,sl->pqrs', R, R, C0, R, R)
        return out
    t_A = time_fn(naive, n_runs=1, n_repeat=2)
    C_ref = naive()
    done(t_A)

    # (B) opt_einsum
    progress("(B) opt_einsum (optimal path, loop over points)")
    expr = 'pi,qj,ijkl,rk,sl->pqrs'
    pi = oe.contract_path(expr, R_all[0], R_all[0], C0, R_all[0], R_all[0])
    def opt():
        out = np.zeros((NP, 3, 3, 3, 3))
        for n in range(NP):
            R = R_all[n]
            out[n] = oe.contract(expr, R, R, C0, R, R, optimize=pi[0])
        return out
    t_B = time_fn(opt, n_runs=1, n_repeat=2)
    err_B = np.max(np.abs(opt() - C_ref))
    done(t_B)

    # (C) JAX
    progress("(C) JAX vmap+jit (batched, sequential optimal path)")
    C_j, R_j = jnp.array(C0), jnp.array(R_all)
    @jit
    def jax_fn(Rb, Ct):
        def rot(R):
            t1 = jnp.einsum('pi,ijkl->pjkl', R, Ct)
            t2 = jnp.einsum('qj,pjkl->pqkl', R, t1)
            t3 = jnp.einsum('rk,pqkl->pqrl', R, t2)
            return jnp.einsum('sl,pqrl->pqrs', R, t3)
        return vmap(rot)(Rb)
    _ = jax_fn(R_j, C_j).block_until_ready()
    def run_jax():
        return jax_fn(R_j, C_j).block_until_ready()
    t_C = time_fn(run_jax, n_runs=3, n_repeat=3)
    err_C = np.max(np.abs(np.array(jax_fn(R_j, C_j)) - C_ref))
    done(t_C)

    fi = extract_flop_info(pi)
    report([
        ("(A) Conventional np.einsum (no path opt)",        t_A, None),
        ("(B) opt_einsum (optimal path, loop)",       t_B, err_B),
        ("(C) JAX vmap+jit (batched + optimal path)", t_C, err_C),
    ], fi)

    return {
        'labels': ['Conventional\nnp.einsum\n(no path opt)',
                    'opt_einsum\n(optimal path,\nloop)',
                    'JAX vmap+jit\n(batched +\noptimal path)'],
        'times':  [t_A, t_B, t_C],
        'errors': [0.0, err_B, err_C],
        'flops_naive': 3.280e4,
        'flops_opt':   1.944e3,
    }



## Benchmark 2: J2 Plasticity Tangent Modulus

**Physical context:** During Newton–Raphson iterations in nonlinear FEA, the algorithmic consistent tangent $\mathbb{C}_{ep}$ must be evaluated at every integration point. For $J_2$ plasticity with isotropic hardening:

$$\mathbb{C}_{ep} = 3K\,\mathbb{I}^{\mathrm{vol}} + 2\mu\,\beta_1\,\mathbb{I}^{\mathrm{dev}} + 2\mu\,\beta_2\,(\hat{\mathbf{n}} \otimes \hat{\mathbf{n}})$$

where the elastic/plastic decision involves an `if/else` branch at each point.

**What is being compared:**
- **(A) Conventional:** Python loop with `if/else` branching per point
- **(B) opt\_einsum:** Same loop, using `oe.contract` for the outer product $\hat{\mathbf{n}} \otimes \hat{\mathbf{n}}$
- **(C) JAX branchless:** Both elastic and plastic tangents computed unconditionally; result selected via `jnp.where` — enables full vectorisation

**Key insight:** Branch elimination contributes ~25× speedup by allowing SIMD execution. Combined with loop elimination (~60×) and JIT fusion (~3×), the total reaches ~181×. This is the largest speedup because constitutive tangent evaluation is where mechanics codes spend most of their assembly time.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CASE 3 : NONLINEAR CONSTITUTIVE TANGENT (J2 PLASTICITY)
# ══════════════════════════════════════════════════════════════════════════════

def run_case_3():
    banner(3, "Nonlinear Constitutive Tangent (J2 Plasticity)")
    print("    Algorithmic tangent C_ep at 10000 integration points")
    print("    Mixed elastic / plastic states")
    print()

    NP = 10000
    np.random.seed(99)

    E, nu = 200e9, 0.3
    Kb = E / (3 * (1 - 2 * nu))
    mu = E / (2 * (1 + nu))
    sy, H = 250e6, 1e9

    sig_trial = np.random.randn(NP, 3, 3) * 300e6
    sig_trial = 0.5 * (sig_trial + sig_trial.transpose(0, 2, 1))

    I2 = np.eye(3)
    I4s = 0.5 * (np.einsum('ik,jl->ijkl', I2, I2) + np.einsum('il,jk->ijkl', I2, I2))
    I4v = (1.0/3.0) * np.einsum('ij,kl->ijkl', I2, I2)
    I4d = I4s - I4v
    Ce  = 3*Kb*I4v + 2*mu*I4d

    def tangent_point(sig):
        s = sig - (np.trace(sig)/3.0)*I2
        seq = np.sqrt(1.5*np.sum(s*s))
        if seq > sy:
            nh = s / seq
            dg = (seq - sy)/(3*mu + H)
            b1 = 1.0 - 3.0*mu*dg/seq
            b2 = 1.0/(1.0 + H/(3.0*mu)) - b1
            return 3*Kb*I4v + 2*mu*b1*I4d + 2*mu*b2*np.einsum('ij,kl->ijkl', nh, nh)
        return Ce.copy()

    progress("(A) Conventional NumPy loop (branching)")
    def naive():
        out = np.zeros((NP, 3, 3, 3, 3))
        for n in range(NP):
            out[n] = tangent_point(sig_trial[n])
        return out
    t_A = time_fn(naive, n_runs=1, n_repeat=2)
    C_ref = naive()
    done(t_A)

    progress("(B) opt_einsum (loop, outer product)")
    def opt():
        out = np.zeros((NP, 3, 3, 3, 3))
        for n in range(NP):
            sig = sig_trial[n]
            s = sig - (np.trace(sig)/3.0)*I2
            seq = np.sqrt(1.5*np.sum(s*s))
            if seq > sy:
                nh = s / seq
                dg = (seq - sy)/(3*mu + H)
                b1 = 1.0 - 3.0*mu*dg/seq
                b2 = 1.0/(1.0 + H/(3.0*mu)) - b1
                out[n] = 3*Kb*I4v + 2*mu*b1*I4d + 2*mu*b2*oe.contract('ij,kl->ijkl', nh, nh)
            else:
                out[n] = Ce
        return out
    t_B = time_fn(opt, n_runs=1, n_repeat=2)
    err_B = np.max(np.abs(opt() - C_ref))
    done(t_B)

    progress("(C) JAX vmap+jit (branchless, fully batched)")
    I2_j  = jnp.eye(3)
    I4v_j, I4d_j, Ce_j = jnp.array(I4v), jnp.array(I4d), jnp.array(Ce)
    sig_j = jnp.array(sig_trial)
    @jit
    def jax_fn(sb):
        def pt(sig):
            s = sig - (jnp.trace(sig)/3.0)*I2_j
            seq = jnp.sqrt(1.5*jnp.sum(s*s))
            ss = jnp.maximum(seq, 1e-30)
            nh = s / ss
            dg = (seq - sy)/(3*mu + H)
            b1 = 1.0 - 3.0*mu*dg/ss
            b2 = 1.0/(1.0 + H/(3.0*mu)) - b1
            Cp = 3*Kb*I4v_j + 2*mu*b1*I4d_j + 2*mu*b2*jnp.einsum('ij,kl->ijkl', nh, nh)
            return jnp.where(seq > sy, Cp, Ce_j)
        return vmap(pt)(sb)
    _ = jax_fn(sig_j).block_until_ready()
    def run_jax():
        return jax_fn(sig_j).block_until_ready()
    t_C = time_fn(run_jax, n_runs=3, n_repeat=3)
    err_C = np.max(np.abs(np.array(jax_fn(sig_j)) - C_ref))
    done(t_C)

    report([
        ("(A) Conventional NumPy loop (branching)",    t_A, None),
        ("(B) opt_einsum (loop, outer prod)",    t_B, err_B),
        ("(C) JAX vmap+jit (branchless batch)",  t_C, err_C),
    ])

    return {
        'labels': ['Conventional\nNumPy loop\n(branching)',
                    'opt_einsum\n(loop,\nouter product)',
                    'JAX vmap+jit\n(branchless,\nbatched)'],
        'times':  [t_A, t_B, t_C],
        'errors': [0.0, err_B, err_C],
    }



## Benchmark 3: High-Order Matrix-Free Operator ($p=4$ Hex)

**Physical context:** In matrix-free FEM, the stiffness operator is never assembled globally. Instead:

$$\mathbf{f}_e = \sum_{g=1}^{N_{GP}} w_g\,\det J_g\; \mathbf{B}_g^T\,\mathbf{D}\,\mathbf{B}_g\,\mathbf{u}_e$$

For a $p=4$ hexahedral element: 125 nodes, 375 DOF, 125 Gauss points, $\mathbf{B}_g \in \mathbb{R}^{6 \times 375}$.

**Dense methods (A–C):**
- **(A) Conventional:** Double loop over elements × Gauss points
- **(B) Fused einsum:** Single five-operand contraction, chunked over elements — lets NumPy's path optimiser choose the evaluation order
- **(C) Batched einsum:** Loop over GP, batch over elements

**Sum-factorisation (D–E):** Exploits the tensor-product structure of hex shape functions. Instead of dense $\mathbf{B}_g \mathbf{u}_e$, the gradient is computed via sequences of 1D contractions: $\mathcal{O}(p^6) \to \mathcal{O}(p^4)$.
- **(D)** Per-element loop
- **(E)** Batched over all elements

**Key insight:** The fused einsum wins at $p=4$ on CPU because BLAS is highly optimised for the resulting intermediate matrix sizes. Sum-factorisation wins at higher $p$ and on GPU.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CASE 4 : HIGH-ORDER MATRIX-FREE OPERATOR APPLICATION
# ══════════════════════════════════════════════════════════════════════════════

def run_case_4():
    banner(4, "High-Order Matrix-Free Operator (p=4 Hex)")
    print("    f_e = sum_g w_g B_g^T D B_g u_e   (matrix-free application)")
    print("    p=4 hex -> 125 nodes, 3000 elements")
    print("    Methods: conventional dense -> fused einsum -> sum-factorization")
    print()

    NE   = 3000
    P    = 4
    N1D  = P + 1
    NNOD = N1D**3
    ND   = NNOD * 3
    NG1D = P + 1
    NG   = NG1D**3
    NS   = 6

    np.random.seed(7)

    E, nu = 200e9, 0.3
    lam = E * nu / ((1 + nu) * (1 - 2 * nu))
    mu  = E / (2 * (1 + nu))
    D   = np.zeros((NS, NS))
    D[:3, :3] = lam
    D[0, 0] = D[1, 1] = D[2, 2] = lam + 2 * mu
    D[3, 3] = D[4, 4] = D[5, 5] = mu

    print("    Generating test data ...", flush=True)
    B_gp  = np.random.randn(NG, NS, ND) * 0.001
    jw    = np.abs(np.random.randn(NE, NG)) * 0.1 + 0.9
    u_all = np.random.randn(NE, ND) * 1e-3

    B1d = np.random.randn(NG1D, N1D) * 0.1
    N1d = np.random.randn(NG1D, N1D) * 0.1
    w1d = np.abs(np.random.randn(NG1D)) + 0.5
    J_diag = np.abs(np.random.randn(NE, 3)) * 0.1 + 0.9
    detJ_sf = J_diag[:, 0] * J_diag[:, 1] * J_diag[:, 2]
    u_3d = np.random.randn(NE, 3, N1D, N1D, N1D) * 1e-3
    w3d = np.einsum('i,j,k->ijk', w1d, w1d, w1d)
    print("    Data ready.\n")

    # (A) Naive
    progress("(A) Conventional loop (correct weighting)")
    def naive():
        f = np.zeros((NE, ND))
        for e in range(NE):
            fe = np.zeros(ND)
            ue = u_all[e]
            for g in range(NG):
                Bg = B_gp[g]
                strain = Bg @ ue
                stress = D @ strain
                fe += Bg.T @ stress * jw[e, g]
            f[e] = fe
        return f
    t_A = time_fn(naive, n_runs=1, n_repeat=1)
    f_ref = naive()
    done(t_A)

    # (B) Fused einsum
    progress("(B) Fused einsum (no element loop, chunked)")
    def fused_chunked(chunk=100):
        f = np.zeros((NE, ND))
        for i in range(0, NE, chunk):
            u_c  = u_all[i:i+chunk]
            jw_c = jw[i:i+chunk]
            f[i:i+chunk] = np.einsum('gsn, st, gtm, em, eg -> en',
                                      B_gp, D, B_gp, u_c, jw_c,
                                      optimize=True)
        return f
    t_B = time_fn(fused_chunked, n_runs=1, n_repeat=1)
    f_B = fused_chunked()
    err_B = np.max(np.abs(f_B - f_ref)) / (np.max(np.abs(f_ref)) + 1e-30)
    done(t_B)

    # (C) Batched einsum (batch elem, GP loop)
    progress("(C) Batched einsum (batch elem, GP loop)")
    def batched_gp_loop():
        f = np.zeros((NE, ND))
        for g in range(NG):
            Bg  = B_gp[g]
            wg  = jw[:, g]
            strain = np.einsum('sd, ed -> es', Bg, u_all)
            stress = np.einsum('st, et -> es', D, strain)
            f += np.einsum('sd, es, e -> ed', Bg, stress, wg)
        return f
    t_C = time_fn(batched_gp_loop, n_runs=1, n_repeat=2)
    f_C = batched_gp_loop()
    err_C = np.max(np.abs(f_C - f_ref)) / (np.max(np.abs(f_ref)) + 1e-30)
    done(t_C)

    # (D) Sum-factorization per-element loop
    progress("(D) Sum-factorization (per-element loop)")
    def sum_factorization():
        f_3d = np.zeros_like(u_3d)
        for e in range(NE):
            Jd = J_diag[e]
            dJ = detJ_sf[e]
            w_sc = w3d * dJ
            for c in range(3):
                uc = u_3d[e, c]
                t1 = np.einsum('ck, ijk -> ijc', N1d, uc)
                t2 = np.einsum('bj, ijc -> ibc', N1d, t1)
                dudx = np.einsum('ai, ibc -> abc', B1d, t2) / Jd[0]
                t1 = np.einsum('ck, ijk -> ijc', N1d, uc)
                t2 = np.einsum('bj, ijc -> ibc', B1d, t1)
                dudy = np.einsum('ai, ibc -> abc', N1d, t2) / Jd[1]
                t1 = np.einsum('ck, ijk -> ijc', B1d, uc)
                t2 = np.einsum('bj, ijc -> ibc', N1d, t1)
                dudz = np.einsum('ai, ibc -> abc', N1d, t2) / Jd[2]
                sx   = mu * dudx * w_sc
                sy_c = mu * dudy * w_sc
                sz   = mu * dudz * w_sc
                t1 = np.einsum('ai, abc -> ibc', B1d, sx) / Jd[0]
                t2 = np.einsum('bj, ibc -> ijc', N1d, t1)
                fc_x = np.einsum('ck, ijc -> ijk', N1d, t2)
                t1 = np.einsum('ai, abc -> ibc', N1d, sy_c) / Jd[1]
                t2 = np.einsum('bj, ibc -> ijc', B1d, t1)
                fc_y = np.einsum('ck, ijc -> ijk', N1d, t2)
                t1 = np.einsum('ai, abc -> ibc', N1d, sz) / Jd[2]
                t2 = np.einsum('bj, ibc -> ijc', N1d, t1)
                fc_z = np.einsum('ck, ijc -> ijk', B1d, t2)
                f_3d[e, c] = fc_x + fc_y + fc_z
        return f_3d
    t_D = time_fn(sum_factorization, n_runs=1, n_repeat=1)
    done(t_D)

    # (E) Sum-factorization batched
    progress("(E) Sum-factorization batched (no element loop)")
    def sum_fact_batched():
        f_out = np.zeros_like(u_3d)
        w_sc = w3d[None, :, :, :] * detJ_sf[:, None, None, None]
        for c in range(3):
            uc = u_3d[:, c]
            t1 = np.einsum('ck, eijk -> eijc', N1d, uc)
            t2 = np.einsum('bj, eijc -> eibc', N1d, t1)
            dudx = np.einsum('ai, eibc -> eabc', B1d, t2)
            dudx = dudx / J_diag[:, 0, None, None, None]
            t1 = np.einsum('ck, eijk -> eijc', N1d, uc)
            t2 = np.einsum('bj, eijc -> eibc', B1d, t1)
            dudy = np.einsum('ai, eibc -> eabc', N1d, t2)
            dudy = dudy / J_diag[:, 1, None, None, None]
            t1 = np.einsum('ck, eijk -> eijc', B1d, uc)
            t2 = np.einsum('bj, eijc -> eibc', N1d, t1)
            dudz = np.einsum('ai, eibc -> eabc', N1d, t2)
            dudz = dudz / J_diag[:, 2, None, None, None]
            sx   = mu * dudx * w_sc
            sy_c = mu * dudy * w_sc
            sz   = mu * dudz * w_sc
            t1 = np.einsum('ai, eabc -> eibc', B1d, sx) / J_diag[:, 0, None, None, None]
            t2 = np.einsum('bj, eibc -> eijc', N1d, t1)
            fc_x = np.einsum('ck, eijc -> eijk', N1d, t2)
            t1 = np.einsum('ai, eabc -> eibc', N1d, sy_c) / J_diag[:, 1, None, None, None]
            t2 = np.einsum('bj, eibc -> eijc', B1d, t1)
            fc_y = np.einsum('ck, eijc -> eijk', N1d, t2)
            t1 = np.einsum('ai, eabc -> eibc', N1d, sz) / J_diag[:, 2, None, None, None]
            t2 = np.einsum('bj, eibc -> eijc', N1d, t1)
            fc_z = np.einsum('ck, eijc -> eijk', B1d, t2)
            f_out[:, c] = fc_x + fc_y + fc_z
        return f_out
    t_E = time_fn(sum_fact_batched, n_runs=2, n_repeat=2)
    done(t_E)

    print()
    print("    -- Dense operator results (A vs B vs C) --")
    report([
        ("(A) Conventional loop (correct weighting)",       t_A, None),
        ("(B) Fused einsum (no elem loop, chunked)",  t_B, err_B),
        ("(C) Batched einsum (batch elem, GP loop)",  t_C, err_C),
    ])
    print("    -- Sum-factorization results (D vs E) --")
    report([
        ("(D) Sum-fact per-element loop",             t_D, None),
        ("(E) Sum-fact batched (no element loop)",     t_E, None),
    ])
    print(f"    Dense conventional (A):               {t_A:.4f} s")
    print(f"    Best dense optimized (B):       {t_B:.4f} s  ({t_A/t_B:.1f}x)")
    print(f"    Sum-fact batched (E):            {t_E:.4f} s  ({t_A/t_E:.1f}x vs A)")
    print()

    return {
        'dense_labels': ['Conventional loop\n(elem + GP)', 'Fused einsum\n(chunked)', 'Batched einsum\n(GP loop)'],
        'dense_times':  [t_A, t_B, t_C],
        'sf_labels':    ['Sum-fact\n(elem loop)', 'Sum-fact\n(batched)'],
        'sf_times':     [t_D, t_E],
        't_naive': t_A,
    }



## Publication Figures

The following cell generates four publication-ready figures (300 DPI, PDF + PNG):

1. **fig\_case2\_rotation** — Anisotropic rotation: wall-clock time + FLOP comparison
2. **fig\_case3\_j2plasticity** — J2 tangent: log-scale timing + speedup decomposition
3. **fig\_case4\_highorder** — High-order: dense methods + sum-factorisation
4. **fig\_summary\_speedups** — Summary bar chart across all benchmarks

Figures use a colorblind-safe palette (Okabe–Ito) and serif fonts for journal compatibility.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  PUBLICATION FIGURES
# ══════════════════════════════════════════════════════════════════════════════

def make_figures(res2, res3, res4, output_dir='.'):
    """Generate publication-quality figures for Cases 2-4."""

    print("\n" + "=" * 72)
    print("  GENERATING PUBLICATION FIGURES")
    print("=" * 72)

    # ── Global style ──
    plt.rcParams.update({
        'font.family':       'serif',
        'font.size':         11,
        'axes.labelsize':    13,
        'axes.titlesize':    14,
        'xtick.labelsize':   10,
        'ytick.labelsize':   11,
        'legend.fontsize':   10,
        'figure.dpi':        300,
        'savefig.dpi':       300,
        'savefig.bbox':      'tight',
        'axes.spines.top':   False,
        'axes.spines.right': False,
        'axes.linewidth':    0.8,
        'xtick.major.width': 0.8,
        'ytick.major.width': 0.8,
    })

    # Colorblind-safe palette (Okabe-Ito)
    C_NAIVE  = '#E69F00'   # orange
    C_OPT    = '#56B4E9'   # sky blue
    C_JAX    = '#009E73'   # teal green
    C_FUSED  = '#56B4E9'
    C_BATCH  = '#CC79A7'   # rose
    C_SF     = '#0072B2'   # dark blue
    C_SFB    = '#009E73'   # teal

    # ══════════════════════════════════════════════════════════════════════
    #  FIGURE 1: Case 2 — Bar chart with speedup annotations
    # ══════════════════════════════════════════════════════════════════════
    print("    Fig 1: Case 2 — Anisotropic tensor rotation ...", flush=True)

    fig1, (ax1a, ax1b) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={'width_ratios': [3, 2]})

    # (a) Wall-clock time
    colors2 = [C_NAIVE, C_OPT, C_JAX]
    bars = ax1a.bar(range(3), res2['times'], color=colors2, width=0.6, edgecolor='#333333', linewidth=0.6)
    ax1a.set_xticks(range(3))
    ax1a.set_xticklabels(res2['labels'], fontsize=9, ha='center')
    ax1a.set_ylabel('Wall-clock time (s)')
    ax1a.set_title('(a) Anisotropic tensor rotation\n'
                    r'$C^\prime_{pqrs} = R_{pi}\,R_{qj}\,C_{ijkl}\,R_{rk}\,R_{sl}$',
                    fontsize=12)

    # Add time labels on bars
    for bar, t in zip(bars, res2['times']):
        if t > 0.05:
            ax1a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                      f'{t:.3f} s', ha='center', va='bottom', fontsize=9, fontweight='bold')
        else:
            ax1a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                      f'{t*1000:.1f} ms', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Add speedup annotation for best method
    sp = res2['times'][0] / res2['times'][2]
    ax1a.annotate(f'{sp:.0f}x\nspeedup',
                  xy=(2, res2['times'][2]),
                  xytext=(2, res2['times'][0] * 0.55),
                  fontsize=12, fontweight='bold', color=C_JAX, ha='center',
                  arrowprops=dict(arrowstyle='->', color=C_JAX, lw=1.5))

    ax1a.set_ylim(0, max(res2['times']) * 1.25)

    # (b) FLOP count comparison (log scale)
    flop_labels = ['Conventional\n(all-at-once)', 'Optimized\n(sequential)']
    flop_vals   = [res2['flops_naive'], res2['flops_opt']]
    flop_colors = [C_NAIVE, C_JAX]

    bars2 = ax1b.bar(range(2), flop_vals, color=flop_colors, width=0.5,
                     edgecolor='#333333', linewidth=0.6)
    ax1b.set_yscale('log')
    ax1b.set_xticks(range(2))
    ax1b.set_xticklabels(flop_labels, fontsize=9)
    ax1b.set_ylabel('FLOPs per contraction')
    ax1b.set_title(r'(b) Contraction path: $O(N^8) \to O(N^5)$', fontsize=12)

    for bar, v in zip(bars2, flop_vals):
        ax1b.text(bar.get_x() + bar.get_width()/2, v * 0.5,
                  f'{v:.0f}', ha='center', va='center', fontsize=9, fontweight='bold',
                  color='white')

    # Scaling labels
    ax1b.text(0, flop_vals[0] * 0.15, r'$O(N^8)$', ha='center', fontsize=11,
              color='white', fontweight='bold')
    ax1b.text(1, flop_vals[1] * 3.0, r'$O(N^5)$', ha='center', fontsize=11,
              color='#333333', fontweight='bold')

    fig1.tight_layout(w_pad=3)
    fig1.savefig(f'{output_dir}/fig_case2_rotation.png')
    fig1.savefig(f'{output_dir}/fig_case2_rotation.pdf')
    print("done")

    # ══════════════════════════════════════════════════════════════════════
    #  FIGURE 2: Case 3 — J2 plasticity tangent
    # ══════════════════════════════════════════════════════════════════════
    print("    Fig 2: Case 3 — J2 plasticity tangent ...", flush=True)

    fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={'width_ratios': [3, 2]})

    # (a) Wall-clock time (log scale due to huge range)
    colors3 = [C_NAIVE, C_OPT, C_JAX]
    bars3 = ax2a.bar(range(3), res3['times'], color=colors3, width=0.6,
                     edgecolor='#333333', linewidth=0.6)
    ax2a.set_yscale('log')
    ax2a.set_xticks(range(3))
    ax2a.set_xticklabels(res3['labels'], fontsize=9, ha='center')
    ax2a.set_ylabel('Wall-clock time (s, log scale)')
    ax2a.set_title('(a)  J2 plasticity tangent modulus\n'
                    r'$\mathbf{C}_{ep}$ at 10,000 integration points',
                    fontsize=12)

    for bar, t in zip(bars3, res3['times']):
        if t > 0.01:
            label = f'{t:.3f} s'
        else:
            label = f'{t*1000:.2f} ms'
        ax2a.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3,
                  label, ha='center', va='bottom', fontsize=9, fontweight='bold')

    sp3 = res3['times'][0] / res3['times'][2]
    ax2a.annotate(f'{sp3:.0f}x\nspeedup',
                  xy=(2, res3['times'][2]),
                  xytext=(1.5, res3['times'][0] * 0.3),
                  fontsize=12, fontweight='bold', color=C_JAX, ha='center',
                  arrowprops=dict(arrowstyle='->', color=C_JAX, lw=1.5))

    # (b) Mechanism decomposition: what contributes to the speedup
    mechanisms = ['Python loop\nelimination\n(vmap)', 'Branch\nelimination\n(jnp.where)',
                  'XLA JIT\ncompilation']
    # Approximate breakdown of the ~200x speedup
    # Loop: ~10000 iterations eliminated → ~50-100x
    # Branching: removes if/else per point → ~2-3x
    # JIT: fuses operations → ~1.5-2x
    mech_vals = [60, 25, 3]  # approximate multiplicative contributions
    mech_colors = ['#D55E00', '#CC79A7', '#0072B2']

    bars_m = ax2b.barh(range(3), mech_vals, color=mech_colors, height=0.5,
                       edgecolor='#333333', linewidth=0.6)
    ax2b.set_yticks(range(3))
    ax2b.set_yticklabels(mechanisms, fontsize=9)
    ax2b.set_xlabel('Approximate contribution factor')
    ax2b.set_title('(b) Speedup decomposition', fontsize=12)
    ax2b.set_xscale('log')

    for bar, v in zip(bars_m, mech_vals):
        ax2b.text(bar.get_width() * 1.15, bar.get_y() + bar.get_height()/2,
                  f'~{v}x', ha='left', va='center', fontsize=10, fontweight='bold')

    fig2.tight_layout(w_pad=3)
    fig2.savefig(f'{output_dir}/fig_case3_j2plasticity.png')
    fig2.savefig(f'{output_dir}/fig_case3_j2plasticity.pdf')
    print("done")

    # ══════════════════════════════════════════════════════════════════════
    #  FIGURE 3: Case 4 — High-order matrix-free (combined figure)
    # ══════════════════════════════════════════════════════════════════════
    print("    Fig 3: Case 4 — High-order matrix-free ...", flush=True)

    fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(10, 4.5))

    # (a) Dense operator methods
    colors4d = [C_NAIVE, C_FUSED, C_BATCH]
    bars4 = ax3a.bar(range(3), res4['dense_times'], color=colors4d, width=0.6,
                     edgecolor='#333333', linewidth=0.6)
    ax3a.set_xticks(range(3))
    ax3a.set_xticklabels(res4['dense_labels'], fontsize=9)
    ax3a.set_ylabel('Wall-clock time (s)')
    ax3a.set_title(r'(a) Dense operator: $\mathbf{f}_e = \sum_g \mathbf{B}_g^T \mathbf{D}\,\mathbf{B}_g\,\mathbf{u}_e$',
                    fontsize=12)

    t_naive_4 = res4['t_naive']
    for bar, t in zip(bars4, res4['dense_times']):
        sp = t_naive_4 / t
        label = f'{t:.3f} s\n({sp:.1f}x)' if sp > 1.05 else f'{t:.3f} s\n(ref)'
        ax3a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                  label, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3a.set_ylim(0, max(res4['dense_times']) * 1.3)

    # (b) Sum-factorization methods
    all_labels = res4['dense_labels'][:1] + res4['sf_labels']  # conventional + SF methods
    all_times  = [res4['dense_times'][0]] + res4['sf_times']
    colors_sf  = [C_NAIVE, C_SF, C_SFB]

    bars_sf = ax3b.bar(range(3), all_times, color=colors_sf, width=0.6,
                       edgecolor='#333333', linewidth=0.6)
    ax3b.set_xticks(range(3))
    ax3b.set_xticklabels(all_labels, fontsize=9)
    ax3b.set_ylabel('Wall-clock time (s)')
    ax3b.set_title('(b) Sum-factorization\n'
                    r'$O(p^6) \to O(p^4)$ via tensor-product 1D ops',
                    fontsize=12)

    for bar, t in zip(bars_sf, all_times):
        sp = t_naive_4 / t
        label = f'{t:.3f} s\n({sp:.1f}x)' if sp > 1.05 else f'{t:.3f} s\n(ref)'
        ax3b.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                  label, ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax3b.set_ylim(0, max(all_times) * 1.3)

    fig3.tight_layout(w_pad=3)
    fig3.savefig(f'{output_dir}/fig_case4_highorder.png')
    fig3.savefig(f'{output_dir}/fig_case4_highorder.pdf')
    print("done")

    # ══════════════════════════════════════════════════════════════════════
    #  FIGURE 4: Summary — All cases on one figure
    # ══════════════════════════════════════════════════════════════════════
    print("    Fig 4: Summary — speedup comparison ...", flush=True)

    fig4, ax4 = plt.subplots(figsize=(8, 4.5))

    case_names = [
        'Anisotropic\ntensor rotation',
        'J2 plasticity\ntangent',
        'High-order\nfused einsum',
        'High-order\nsum-factorization'
    ]
    speedups = [
        res2['times'][0] / res2['times'][2],       # Case 2: JAX
        res3['times'][0] / res3['times'][2],       # Case 3: JAX
        res4['t_naive'] / min(res4['dense_times'][1:]),  # Case 4 dense best
        res4['t_naive'] / res4['sf_times'][-1],    # Case 4 sum-fact batched
    ]
    bar_colors = [C_JAX, C_JAX, C_FUSED, C_SFB]

    bars_sum = ax4.barh(range(len(case_names)), speedups, color=bar_colors,
                        height=0.55, edgecolor='#333333', linewidth=0.6)

    ax4.set_yticks(range(len(case_names)))
    ax4.set_yticklabels(case_names, fontsize=10)
    ax4.set_xlabel('Speedup over conventional implementation (x)', fontsize=12)
    ax4.set_title('Speedup from optimized tensor contraction strategies', fontsize=13)
    ax4.set_xscale('log')
    ax4.set_xlim(1, max(speedups) * 2)

    # 1x reference line
    ax4.axvline(x=1, color='#999999', linestyle='--', linewidth=0.8, zorder=0)

    for bar, sp in zip(bars_sum, speedups):
        ax4.text(bar.get_width() * 1.12, bar.get_y() + bar.get_height()/2,
                 f'{sp:.1f}x', ha='left', va='center', fontsize=11, fontweight='bold')

    fig4.tight_layout()
    fig4.savefig(f'{output_dir}/fig_summary_speedups.png')
    fig4.savefig(f'{output_dir}/fig_summary_speedups.pdf')
    print("done")

    plt.close('all')
    print()
    print(f"    Figures saved to: {output_dir}/")
    print(f"      fig_case2_rotation.png / .pdf")
    print(f"      fig_case3_j2plasticity.png / .pdf")
    print(f"      fig_case4_highorder.png / .pdf")
    print(f"      fig_summary_speedups.png / .pdf")
    print()


# ══════════════════════════════════════════════════════════════════════════════

## Run Everything

The cell below executes all three benchmarks and generates the figures. Total runtime: ~1–2 minutes on Colab CPU.

In [ ]:
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    print("=" * 72)
    print("  TENSOR CONTRACTION BENCHMARK FOR COMPUTATIONAL MECHANICS")
    print("  Methods : Conventional loop  ·  Fused einsum  ·  JAX vmap+jit")
    print("  Platform: CPU only")
    print("=" * 72)

    t0 = time.time()

    res2 = run_case_2()
    res3 = run_case_3()
    res4 = run_case_4()

    elapsed = time.time() - t0

    print("=" * 72)
    print(f"  BENCHMARKS COMPLETE — {elapsed:.1f} s")
    print("=" * 72)

    # ── Generate figures ──
    import os
    out_dir = os.environ.get('OUTPUT_DIR', '.')
    make_figures(res2, res3, res4, output_dir=out_dir)

    print("=" * 72)
    print("  ALL DONE.")
    print("=" * 72)